# 09 • Multimodal RAG จากภาพตัวอย่าง
ดาวน์โหลดภาพจาก GitHub อัตโนมัติ → LLM อธิบาย → embed คำบรรยาย → บันทึกใน runtime → ถามคำถามและค้นภาพกลับมาตอบ

ใช้ภาพตัวอย่างจาก GitHub คำบรรยายใช้ค้นภาพ ส่วนคำตอบสุดท้ายอ่านจากภาพที่บันทึกไว้ด้วย

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/biodatlab/sritrang-ai-workshop/blob/main/09_multimodal_rag.ipynb)


## ติดตั้ง packages
รันจากบนลงล่าง ใส่ API key ในขั้นตอน 1 ดาวน์โหลดภาพจาก GitHub อัตโนมัติในขั้นตอน 2 และพิมพ์คำถามในขั้นตอน 5


In [ ]:
%pip install -q openai numpy pillow


## 1. ตั้งค่า API


In [ ]:
from openai import OpenAI
from pathlib import Path
from PIL import Image
from IPython.display import display
import base64
import io
import json
import numpy as np

OPENAI_API_KEY = "your_api_key"  # แทนที่ด้วย API key จริงของคุณ
client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4.1-mini"
EMBED_MODEL = "text-embedding-3-small"
folder = Path("image_index")
folder.mkdir(exist_ok=True)

def image_content(path):
    encoded = base64.b64encode(Path(path).read_bytes()).decode()
    return {"type": "input_image", "image_url": "data:image/png;base64," + encoded}


## 2. ดาวน์โหลดภาพจาก GitHub
Notebook ดาวน์โหลด `sritrang_training_sample.png` จาก GitHub อัตโนมัติ ไม่ต้องอัปโหลดไฟล์ ระบบเก็บภาพไว้ใน `image_index` ของ runtime


In [ ]:
from urllib.request import urlopen
from urllib.parse import urlparse, unquote

IMAGE_URLS = [
    "https://raw.githubusercontent.com/biodatlab/sritrang-ai-workshop/main/assets/sritrang_training_sample.png",
]
image_paths = []
original_names = {}
for i, url in enumerate(IMAGE_URLS, start=1):
    name = unquote(Path(urlparse(url).path).name)
    with urlopen(url, timeout=60) as response:
        data = response.read()
    path = folder / f"image_{i}.png"
    with Image.open(io.BytesIO(data)) as image:
        image.convert("RGB").save(path)
    image_paths.append(path)
    original_names[path.name] = name
    display(Image.open(path))


## 3. ให้ LLM อธิบายภาพ
สร้างคำบรรยายเพื่อใช้ค้นหา โดยเก็บชื่อไฟล์คู่กับคำบรรยาย


In [ ]:
records = []
for path in image_paths:
    response = client.responses.create(
        model=MODEL,
        input=[{"role": "user", "content": [
            {"type": "input_text", "text": "อธิบายภาพนี้เป็นภาษาไทยเพื่อใช้ค้นหา ระบุข้อความ ตัวเลข ป้ายชื่อ และความสัมพันธ์ที่เห็น หากอ่านไม่ชัดให้บอก ห้ามเดา"},
            image_content(path),
        ]}],
        max_output_tokens=500,
    )
    records.append({"file": path.name, "original_name": original_names[path.name],
                    "caption": response.output_text})
    print(original_names[path.name], "\n", response.output_text, "\n")


## 4. Embed คำบรรยายและบันทึก index
ภาพอยู่ใน `image_index/` ส่วนคำบรรยายและ embeddings บันทึกเป็นไฟล์ในโฟลเดอร์เดียวกัน เพื่อให้ขั้นตอนสุดท้ายโหลดกลับมาใช้


In [ ]:
result = client.embeddings.create(model=EMBED_MODEL, input=[r["caption"] for r in records])
vectors = np.array([item.embedding for item in result.data])
vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
(folder / "records.json").write_text(
    json.dumps({"embedding_model": EMBED_MODEL, "records": records}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
np.save(folder / "vectors.npy", vectors)
print("บันทึกภาพ คำบรรยาย และ embeddings ใน", folder)


## 5. ถามเกี่ยวกับภาพที่บันทึกไว้
พิมพ์คำถามของคุณ เช่น ถามข้อความในภาพ ตำแหน่งในผัง หรือตัวเลขในรายงาน
ระบบโหลด index จากไฟล์ embed คำถาม ค้นภาพที่เกี่ยวข้อง แล้วส่งภาพนั้นให้ LLM ตอบ

หากมีเพียงภาพเดียว ระบบจะใช้ภาพนั้นเป็นหลักฐาน รันเฉพาะ cell นี้ซ้ำเพื่อถามต่อ ไม่ต้องสร้างคำบรรยายและ embeddings ของภาพใหม่


In [ ]:
def ask_about_image(question):
    saved = json.loads((folder / "records.json").read_text(encoding="utf-8"))
    saved_vectors = np.load(folder / "vectors.npy", allow_pickle=False)
    query_result = client.embeddings.create(model=saved["embedding_model"], input=[question])
    query = np.array(query_result.data[0].embedding)
    scores = saved_vectors @ (query / np.linalg.norm(query))
    best = saved["records"][int(np.argmax(scores))]
    selected_path = folder / best["file"]
    print("ภาพที่ใช้ตอบ:", best["original_name"])
    display(Image.open(selected_path))
    response = client.responses.create(
        model=MODEL,
        input=[{"role": "user", "content": [
            {"type": "input_text", "text": f"คำถาม: {question}\nคำบรรยายสำหรับค้นหา: {best['caption']}\nตอบภาษาไทยจากภาพนี้เท่านั้น ระบุชื่อไฟล์ {best['original_name']} หากภาพไม่มีข้อมูลที่ถามหรืออ่านไม่ชัด ให้บอกว่าไม่พบข้อมูล ห้ามเดา"},
            image_content(selected_path),
        ]}],
        max_output_tokens=700,
    )
    print(response.output_text)

question = input("คุณอยากถามอะไรเกี่ยวกับภาพ? ")
ask_about_image(question)


## เสร็จแล้ว
เปลี่ยนคำถามโดยรันขั้นตอน 5 ซ้ำ ภาพไม่ได้ถูกจำถาวรในโมเดล แต่ถูกโหลดจากไฟล์แล้วส่งให้ LLM ในแต่ละคำถาม
